# Visual target layer ablation
Trains the baseline against the CLS token of CLIP ViT-B/32 blocks 4, 6, 8, 10 and 12, and against the final projected embedding. Reports within-subject and cross-subject retrieval, plus expected k-way accuracy (k = 2, 5, 10).

Uses the cache built by `02_loso.ipynb`.

## Setup

In [ ]:
# Setup: mount Drive, paths, settings
import os, sys, gc, json, math, time, random, shutil, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")   # results and checkpoints
DCACHE = f"{DRIVE}/cache"                                                     # preprocessed EEG and image features
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")                      # raw downloads
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")                        # local copy of the cache
for d in [DRIVE, DCACHE, ROOT, LOCAL]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "openai"
LR, WD = 3e-4, 1e-4

free_gb = shutil.disk_usage(DRIVE).free / 1e9
have = sum(os.path.exists(f"{DCACHE}/{s}_train_z16.npy") for s in ALL_SUBJECTS)
print(f"Drive free: {free_gb:.1f} GB | subjects already saved in Drive: {have}/10")
if free_gb < 0.55 * (10 - have) + 0.2:
    print("warning: low Drive space (about 0.53 GB per subject is needed)")

import numpy as np
for pkg, pip_name in [("open_clip", "open_clip_torch"), ("huggingface_hub", "huggingface_hub"), ("pandas", "pandas")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def drive_ok(path, shape0=None):
    try:
        a = np.load(path, mmap_mode="r"); return shape0 is None or a.shape[0] == shape0
    except Exception:
        return False


## Layer features

In [ ]:
# Visual targets from intermediate CLIP blocks (CLS token), cached
LAYERS = [4, 6, 8, 10, 12]                       # transformer blocks of ViT-B/32 (CLS token, 768-d)
lay_path = f"{DCACHE}/feats_layers_{CLIP_ARCH}_{CLIP_PRETRAINED}.pt"
try:
    LFEAT = torch.load(lay_path); print("layer features loaded")
except Exception:
    import tarfile, open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRETRAINED)
    clip_model = clip_model.to(device).eval()
    blocks = clip_model.visual.transformer.resblocks
    batch_first = getattr(clip_model.visual.transformer, "batch_first", True)
    grabbed = {}
    def hook(k):
        def f(mod, inp, out): grabbed[k] = (out[:, 0] if batch_first else out[0]).float()
        return f
    handles = [blocks[k - 1].register_forward_hook(hook(k)) for k in LAYERS]
    class ImgDS(Dataset):
        def __init__(self, paths): self.paths = paths
        def __len__(self): return len(self.paths)
        def __getitem__(self, i): return preprocess(Image.open(self.paths[i]).convert("RGB"))
    @torch.no_grad()
    def encode(paths):
        out = {k: [] for k in LAYERS}; out["final"] = []
        for x in DataLoader(ImgDS(paths), batch_size=128, num_workers=2):
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                fin = clip_model.encode_image(x.to(device)).float()
            out["final"].append(F.normalize(fin, dim=-1).cpu())
            for k in LAYERS: out[k].append(F.normalize(grabbed[k], dim=-1).cpu())
        return {k: torch.cat(v).half() for k, v in out.items()}
    t0 = time.time()
    tr, te = encode([idx_train[f] for f in train_files]), encode([idx_test[f] for f in test_files])
    for h in handles: h.remove()
    LFEAT = {k: {"train": tr[k], "test": te[k]} for k in tr}
    LFEAT["n_concepts"] = len(set(meta["train_img_concepts"]))
    torch.save(LFEAT, lay_path + ".tmp"); os.replace(lay_path + ".tmp", lay_path)
    del clip_model; torch.cuda.empty_cache() if device == "cuda" else None
    print(f"layer features computed in {time.time()-t0:.0f}s")
N_TR = LFEAT["final"]["train"].shape[0]


## Training per target

In [ ]:
# Train the baseline against each visual target.
# Within: train and test on the same subject. Cross: the same model on CROSS_TEST, no adaptation.
SUBJECTS   = ["sub-01"]
CROSS_TEST = "sub-02"
TARGETS    = LAYERS + ["final"]
SEED, EPOCHS, PATIENCE, BATCH = 0, 40, 10, 256

class EEGEncoder(nn.Module):
    def __init__(self, n_ch, n_t, out_dim, k=40, drop=0.5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, k, (1, 25)), nn.AvgPool2d((1, 51), (1, 5)), nn.BatchNorm2d(k), nn.ELU(),
            nn.Conv2d(k, k, (n_ch, 1)), nn.BatchNorm2d(k), nn.ELU(), nn.Dropout(drop))
        with torch.no_grad(): flat = self.backbone(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(flat, out_dim))
        self.res = nn.Sequential(nn.GELU(), nn.Linear(out_dim, out_dim), nn.Dropout(drop))
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        h = self.proj(self.backbone(x.unsqueeze(1)))
        return F.normalize(self.norm(h + self.res(h)), dim=-1)

@torch.no_grad()
def embed(model, X, bs=512):
    model.eval(); return torch.cat([model(X[i:i + bs].float()) for i in range(0, len(X), bs)])
@torch.no_grad()
def retrieval(z, targets, ks=(1, 5)):
    sim = z @ targets.T
    rank = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {f"top{k}": (rank < k).float().mean().item() * 100 for k in ks}
@torch.no_grad()
def kway(z, targets, ks=(2, 5, 10)):
    # expected k-way accuracy: true image plus k-1 random distractors (exact, from ranks)
    sim = z @ targets.T
    r = (sim > sim.diag().unsqueeze(1)).sum(1).double()          # distractors ranked above the target
    n = sim.shape[1] - 1
    out = {}
    for k in ks:
        lg = lambda a, b: torch.lgamma(a + 1) - torch.lgamma(b + 1) - torch.lgamma(a - b + 1)
        p = torch.exp(lg(n - r, torch.tensor(k - 1.0, dtype=torch.double, device=r.device)) - lg(torch.tensor(float(n), dtype=torch.double), torch.tensor(k - 1.0, dtype=torch.double)).to(r.device))
        p = torch.where(n - r >= k - 1, p, torch.zeros_like(p))
        out[f"{k}way"] = p.mean().item() * 100
    return out
def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

PER = N_TR // LFEAT["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)
VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device); TRN = torch.from_numpy(np.where(~IS_VAL)[0]).to(device)
Xc = torch.from_numpy(np.load(local_copy(f"{CROSS_TEST}_test_z16.npy"))).float().to(device)

def run(sub, target):
    set_seed(SEED); t0 = time.time()
    X = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    T_tr, T_te = LFEAT[target]["train"].float().to(device), LFEAT[target]["test"].float().to(device)
    model = EEGEncoder(X.shape[1], X.shape[2], T_tr.shape[1]).to(device)
    logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=device))
    opt = torch.optim.AdamW(list(model.parameters()) + [logit_scale], lr=LR, weight_decay=WD)
    best, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(EPOCHS):
        model.train(); perm = TRN[torch.randperm(len(TRN), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            logits = logit_scale.exp().clamp(max=100) * model(X[b].float()) @ T_tr[b].T
            lab = torch.arange(len(b), device=device)
            loss = (F.cross_entropy(logits, lab) + F.cross_entropy(logits.T, lab)) / 2
            opt.zero_grad(); loss.backward(); opt.step()
        va = retrieval(embed(model, X[VAL]), T_tr[VAL])["top5"]
        if va > best: best, best_ep, bad, best_state = va, ep + 1, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else: bad += 1
        if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    zt, zc = embed(model, Xt), embed(model, Xc)
    w, c, wk, ck = retrieval(zt, T_te), retrieval(zc, T_te), kway(zt, T_te), kway(zc, T_te)
    return {"subject": sub, "target": f"layer {target}" if target != "final" else "final (projected)", "best_epoch": best_ep,
            "within_top1": w["top1"], "within_top5": w["top5"], f"cross_{CROSS_TEST}_top1": c["top1"], f"cross_{CROSS_TEST}_top5": c["top5"],
            **{f"within_{k}": v for k, v in wk.items()}, **{f"cross_{k}": v for k, v in ck.items()},
            "minutes": round((time.time() - t0) / 60, 1)}

for sub in SUBJECTS:
    for tg in TARGETS:
        path = f"{DRIVE}/layer_{sub}_{tg}.json"
        if os.path.exists(path): print(f"{sub} {tg}: done"); continue
        r = run(sub, tg); json.dump(r, open(path, "w"))
        print(f"{sub} {r['target']}: within top1 {r['within_top1']:.1f}% | cross {CROSS_TEST} top1 {r[f'cross_{CROSS_TEST}_top1']:.1f}%  ({r['minutes']} min)", flush=True)

rows = [json.load(open(f"{DRIVE}/layer_{s}_{t}.json")) for s in SUBJECTS for t in TARGETS if os.path.exists(f"{DRIVE}/layer_{s}_{t}.json")]
lay = pd.DataFrame(rows); lay.to_csv(f"{DRIVE}/results_layers.csv", index=False)
print(lay.round(1).to_string(index=False))
